# Model A — PlantVillage lab baseline only

Trains on lab images only. Useful to measure domain gap vs field Model B.

**Not for production.** Deploy Model B: `model_b_combined.keras` from `train_model_b_kaggle.ipynb`.


In [ ]:
import os
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input
from sklearn.metrics import precision_recall_fscore_support

# Lab-only baseline (old "Model A"). Field Model B: train_model_b_kaggle.ipynb
BUILD_MODEL = True
MODEL_PATH = '/kaggle/working/model_a_pv_only.keras'
IMG_SIZE, BATCH = (224, 224), 32

print('=== Model A: PlantVillage lab baseline ===')

# Prefer Kaggle Input for PlantVillage
PV_TRAIN = PV_VALID = None
base = '/kaggle/input'
if os.path.isdir(base):
    for entry in sorted(os.listdir(base)):
        if any(h in entry.lower() for h in ('new-plant-diseases', 'plant-diseases', 'plantvillage')):
            root = os.path.join(base, entry)
            for r, dirs, _ in os.walk(root):
                if 'train' in dirs and 'valid' in dirs:
                    PV_TRAIN, PV_VALID = os.path.join(r, 'train'), os.path.join(r, 'valid')
                    print('PlantVillage from Input:', r)
                    break
        if PV_TRAIN:
            break
if PV_TRAIN is None:
    !kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /kaggle/working/data --unzip
    PV_ROOT = '/kaggle/working/data/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)'
    PV_TRAIN = os.path.join(PV_ROOT, 'train')
    PV_VALID = os.path.join(PV_ROOT, 'valid')

# PlantDoc test (optional field gap check)
PD_DIR = '/kaggle/working/plantdoc'
if not os.path.exists(PD_DIR):
    !git clone -q https://github.com/pratikkayal/PlantDoc-Dataset.git {PD_DIR}
PD_TEST = os.path.join(PD_DIR, 'test')

plantdoc_to_plantvillage = {
    'Apple Scab Leaf': 'Apple___Apple_scab', 'Apple leaf': 'Apple___healthy',
    'Apple rust leaf': 'Apple___Cedar_apple_rust', 'Bell_pepper leaf': 'Pepper,_bell___healthy',
    'Bell_pepper leaf spot': 'Pepper,_bell___Bacterial_spot', 'Blueberry leaf': 'Blueberry___healthy',
    'Cherry leaf': 'Cherry_(including_sour)___healthy',
    'Corn Gray leaf spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn leaf blight': 'Corn_(maize)___Northern_Leaf_Blight', 'Corn rust leaf': 'Corn_(maize)___Common_rust_',
    'Peach leaf': 'Peach___healthy', 'Potato leaf early blight': 'Potato___Early_blight',
    'Potato leaf late blight': 'Potato___Late_blight', 'Raspberry leaf': 'Raspberry___healthy',
    'Soyabean leaf': 'Soybean___healthy', 'Squash Powdery mildew leaf': 'Squash___Powdery_mildew',
    'Strawberry leaf': 'Strawberry___healthy', 'Tomato Early blight leaf': 'Tomato___Early_blight',
    'Tomato Septoria leaf spot': 'Tomato___Septoria_leaf_spot', 'Tomato leaf': 'Tomato___healthy',
    'Tomato leaf bacterial spot': 'Tomato___Bacterial_spot', 'Tomato leaf late blight': 'Tomato___Late_blight',
    'Tomato leaf mosaic virus': 'Tomato___Tomato_mosaic_virus',
    'Tomato leaf yellow virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato mold leaf': 'Tomato___Leaf_Mold',
    'Tomato two spotted spider mites leaf': 'Tomato___Spider_mites Two-spotted_spider_mite',
    'grape leaf': 'Grape___healthy', 'grape leaf black rot': 'Grape___Black_rot',
}

class_names = sorted([d for d in os.listdir(PV_TRAIN) if os.path.isdir(os.path.join(PV_TRAIN, d))])
NUM_CLASSES = len(class_names)
assert NUM_CLASSES == 38

PD_TEST_RELABELED = '/kaggle/working/plantdoc-test-relabeled'
if os.path.exists(PD_TEST_RELABELED):
    shutil.rmtree(PD_TEST_RELABELED)
os.makedirs(PD_TEST_RELABELED, exist_ok=True)
for cls in class_names:
    os.makedirs(os.path.join(PD_TEST_RELABELED, cls), exist_ok=True)
for folder, cls in plantdoc_to_plantvillage.items():
    src_dir = os.path.join(PD_TEST, folder)
    if not os.path.isdir(src_dir):
        continue
    for fname in os.listdir(src_dir):
        s = os.path.abspath(os.path.join(src_dir, fname))
        dst = os.path.join(PD_TEST_RELABELED, cls, f'pd_{fname}')
        if not os.path.lexists(dst):
            os.symlink(s, dst)

aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

def make_ds(directory, shuffle=True):
    ds = tf.keras.utils.image_dataset_from_directory(
        directory, image_size=IMG_SIZE, batch_size=BATCH, shuffle=shuffle,
        seed=42, class_names=class_names)
    if shuffle:
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

pv_train_ds = make_ds(PV_TRAIN)
pv_valid_ds = make_ds(PV_VALID, shuffle=False)
pd_test_ds = make_ds(PD_TEST_RELABELED, shuffle=False)

def build_model():
    base = EfficientNetB0(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    fine_tune_at = len(base.layers) - 40
    base.trainable = False
    model = tf.keras.Sequential([
        base, tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(512, activation='relu'), tf.keras.layers.Dropout(0.45),
        tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model, base, fine_tune_at

if BUILD_MODEL:
    model, base, fine_tune_at = build_model()
    model.fit(pv_train_ds, validation_data=pv_valid_ds, epochs=10)
    base.trainable = True
    for layer in base.layers[:fine_tune_at]:
        layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(pv_train_ds, validation_data=pv_valid_ds, epochs=5)
    model.save(MODEL_PATH)
    print('Saved', MODEL_PATH)
else:
    model = tf.keras.models.load_model(MODEL_PATH)

_, pv_acc = model.evaluate(pv_valid_ds, verbose=0)
_, pd_acc = model.evaluate(pd_test_ds, verbose=0)
print(f'Model A  PV Acc={pv_acc*100:.1f}%  PlantDoc Acc={pd_acc*100:.1f}%')
print('This is the lab baseline only. Deploy field Model B from train_model_b_kaggle.ipynb -> model_b_combined.keras')

with open('/kaggle/working/class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)
json.dump({'model': 'A_lab_baseline', 'pv_acc': float(pv_acc), 'pd_acc': float(pd_acc)},
          open('/kaggle/working/model_a_report.json', 'w'), indent=2)
